In [32]:
import pandas as pd

In [ ]:
stats_list = [
    "fisher_IPS_20_vs_26_results",
    "fisher_Nonprog_IPS_20_vs_26_results",
    "fisher_Prog_IPS_20_vs_26_results"
    "fisher_IPS_N_vs_P_results",
    "fisher_year_20_IPS_N_vs_P_results",
    "fisher_year_26_IPS_N_vs_P_results",
]


for stats_file in stats_list:

    print(f'stats_out/{stats_file}.txt')
    
    stats_df = pd.read_csv(f'/home/dcm/250513Pat_250728Pat_250930Pat/stats_out/{stats_file}.txt', sep = '\t')
    stats_df = stats_df[stats_df['FDR_BH'] < 0.05]

    print(stats_df)
    
    dfm = pd.DataFrame()
    
    project_list = [
        '/home/dcm/250513Pat',
        '/home/dcm/250728Pat',
        '/home/dcm/250930Pat'
        ]
    
    for project in project_list:
    
        file_df = pd.read_csv(f'{project}/files.txt', sep='\t')
        
        for folder, file in zip(file_df['folder'], file_df['file']):
    
            df_IPS = pd.read_csv(f'/home/dcm/250513Pat_250728Pat_250930Pat/IPS_out/{file}_IPS_filter.txt', sep = '\t', usecols = ['A','L'])
    
            mask_IPS = df_IPS['L'].isin(stats_df['otu'])
            
            # Apply the mask to dfA to filter the rows
            df_IPS = df_IPS[mask_IPS]
    
            df_IPS = df_IPS.drop_duplicates()
            
            dfm = pd.concat([dfm, df_IPS], ignore_index=True)
    
            dfm['stats_compare'] = f'IPS_{stats_file}'
    
    dfm.to_csv(f'stats_out/{stats_file}_annotations.txt', sep = '\t', index = False)   

In [28]:
#merge stats out files

dfm = pd.DataFrame()

stats_list = [
    "fisher_IPS_20_vs_26_results",
    "fisher_Nonprog_IPS_20_vs_26_results",
    "fisher_Prog_IPS_20_vs_26_results",  # missing comma fixed
    "fisher_IPS_N_vs_P_results",
    "fisher_year_20_IPS_N_vs_P_results",
    "fisher_year_26_IPS_N_vs_P_results",
]

for stats_file in stats_list:
    try:
        stats_df = pd.read_csv(
            f"stats_out/{stats_file}_annotations.txt",
            sep="\t", usecols = ['A', 'L']
        )

        stats_df = stats_df.rename(columns={"A":"feature_id","L": f"{stats_file}"})
        
        print(f"Loaded: stats_out/{stats_file}_annotations.txt")

        # first dataframe assignment
        if dfm.empty:
            dfm = stats_df
        else:
            dfm = pd.merge(dfm, stats_df, on="feature_id", how="outer")

    except Exception as e:
        print(f"Could not merge/read {stats_file}: {e}")

dfm.to_csv('stats_out/signif_IPS_feature_ids.txt', sep = '\t', index=False)

Loaded: stats_out/fisher_IPS_20_vs_26_results_annotations.txt
Loaded: stats_out/fisher_Nonprog_IPS_20_vs_26_results_annotations.txt
Loaded: stats_out/fisher_Prog_IPS_20_vs_26_results_annotations.txt
Loaded: stats_out/fisher_IPS_N_vs_P_results_annotations.txt
Loaded: stats_out/fisher_year_20_IPS_N_vs_P_results_annotations.txt
Loaded: stats_out/fisher_year_26_IPS_N_vs_P_results_annotations.txt


In [31]:
df_IPS = pd.read_csv('stats_out/signif_IPS_feature_ids.txt', sep = '\t', dtype=str)

dfm = pd.DataFrame()

project_list = [
    '/home/dcm/250513Pat',
    '/home/dcm/250728Pat',
    '/home/dcm/250930Pat'
    ]

for project in project_list:

    file_df = pd.read_csv(f'{project}/files.txt', sep='\t')
    
    for folder, file in zip(file_df['folder'], file_df['file']):
        
        df = pd.read_csv(f'/home/dcm/250513Pat_250728Pat_250930Pat/VFDB_diamond/{file}_VFDB_out_meta.txt', sep = '\t', dtype=str)

        df = pd.merge(df, df_IPS, on = 'feature_id', how = 'inner')

        dfm = pd.concat([dfm, df], ignore_index=True)

dfm.to_csv('stats_out/annotations_signif_IPS_feature_ids.txt', sep = '\t', index=False)


In [34]:
df_IPS = pd.read_csv('stats_out/signif_IPS_feature_ids.txt', sep = '\t', dtype=str)

dfm = pd.DataFrame()

project_list = [
    '/home/dcm/250513Pat',
    '/home/dcm/250728Pat',
    '/home/dcm/250930Pat'
    ]

for project in project_list:

    file_df = pd.read_csv(f'{project}/files.txt', sep='\t')
    
    for folder, file in zip(file_df['folder'], file_df['file']):
        
        df = pd.read_csv(f'/home/dcm/250513Pat_250728Pat_250930Pat/VFDB_diamond/{file}_VFDB_out_meta.txt', sep = '\t', dtype=str)

        df = pd.merge(df, df_IPS, on = 'feature_id', how = 'left')

        dfm = pd.concat([dfm, df], ignore_index=True)

dfm.to_csv('stats_out/all_annotations_signif_IPS_feature_ids.txt', sep = '\t', index=False)